# Create DriveThru Ordering Agent with Voice Live Settings

**One-time setup** — run this notebook after provisioning Azure resources via `resource-setup.azcli`.

This creates the DriveThru ordering agent in your Azure AI Foundry project **with Voice Live
session configuration stored in agent metadata**. The agent owns all instructions, personality,
tools, and voice settings.

**Approach**: Uses `PromptAgentDefinition` with `FunctionTool` objects and chunked Voice Live
metadata (following the [Microsoft Voice Live Agents quickstart](https://learn.microsoft.com/en-us/azure/ai-services/speech-service/voice-live-agents-quickstart)).

**Prerequisites:**
- Azure resources provisioned (`platform/aif-resources/resource-setup.azcli`)
- `.env` populated with `PROJECT_ENDPOINT`, `AGENT_NAME`, `MODEL_DEPLOYMENT_NAME`
- `az login` completed (AzureCliCredential) — or a browser will open for interactive login

**After running:** Copy the printed `AGENT_NAME` and `AGENT_VERSION` into your `.env` file,
then run the validation script: `python -m src.agents.voice_agent_validation`

In [1]:
# Uncomment to install dependencies (run once)
# %pip install azure-ai-projects>=2.0.0 azure-identity python-dotenv

In [2]:
import os
import sys
import json
from dotenv import load_dotenv
from azure.identity import AzureCliCredential, InteractiveBrowserCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition

# Ensure repo root is on the path so we can import src.*
repo_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from src.agent_tools import ALL_TOOLS
from src.agents.create_voice_agent import (
    chunk_config,
    reassemble_config,
    DRIVETHRU_INSTRUCTIONS,
    VOICE_LIVE_CONFIG,
)

# Load environment variables from repo root .env
load_dotenv(os.path.join(repo_root, ".env"), override=True)

project_endpoint = os.environ["PROJECT_ENDPOINT"]
# AzureCliCredential works when 'az login' has been run.
# Falls back to interactive browser login if CLI is unavailable.
try:
    credential = AzureCliCredential()
    credential.get_token("https://management.azure.com/.default")  # verify it works
    print("✅ Authenticated via Azure CLI")
except Exception:
    print("⚠️  Azure CLI credential unavailable — opening browser login...")
    credential = InteractiveBrowserCredential()

project_client = AIProjectClient(
    endpoint=project_endpoint,
    credential=credential,
)
print(f"Connected to: {project_endpoint}")

c:\repos\voiceliveapi-drivethru-agent-demo\.venv\Lib\site-packages\agent_framework\_skills.py:121: ExperimentalWarning: [SKILLS] SkillResource is experimental and may change or be removed in future versions without notice.
c:\repos\voiceliveapi-drivethru-agent-demo\.venv\Lib\site-packages\agent_framework\_harness\_memory.py:651: ExperimentalWarning: [HARNESS] MemoryStore is experimental and may change or be removed in future versions without notice.


✅ Authenticated via Azure CLI
Connected to: https://aif-voiceliveapi-demo.services.ai.azure.com/api/projects/proj-voiceliveapi-demo


In [ ]:
# ── Agent configuration ──────────────────────────────────────────────────────
#
# The agent owns: instructions, personality, FunctionTool definitions,
# and Voice Live session settings (voice, VAD, noise reduction, echo cancellation).
# The relay backend only streams audio — zero function calling on the relay side.

AGENT_NAME = os.environ.get("AGENT_NAME", "drivethru-ordering-agent")
MODEL_DEPLOYMENT_NAME = os.environ.get("MODEL_DEPLOYMENT_NAME", "")

print(f"Agent name   : {AGENT_NAME}")
print(f"Model        : {MODEL_DEPLOYMENT_NAME}")
print(f"Instructions : {len(DRIVETHRU_INSTRUCTIONS)} chars")
print(f"Tools        : {[t.name for t in ALL_TOOLS]}")
print(f"\nVoice Live config:")
print(json.dumps(VOICE_LIVE_CONFIG, indent=2))

Agent name   : drivethru-ordering-agent
Model        : gpt-5.4
Instructions : 1488 chars
Tools        : ['get_menu_items', 'submit_order', 'get_order_analytics']

Voice Live config:
{
  "microsoft.voice-live.enabled": "true",
  "session": {
    "voice": {
      "name": "en-US-Ava:DragonHDLatestNeural",
      "type": "azure-standard",
      "temperature": 0.8
    },
    "inputAudioTranscription": {
      "model": "azure-speech",
      "language": "nl-NL,en-US"
    },
    "turnDetection": {
      "type": "azure_semantic_vad",
      "endOfUtteranceDetection": {
        "model": "semantic_detection_v1_multilingual"
      }
    },
    "inputAudioNoiseReduction": {
      "type": "azure_deep_noise_suppression"
    },
    "inputAudioEchoCancellation": {
      "type": "server_echo_cancellation"
    },
    "fillerResponse": {
      "type": "llm_filler",
      "triggers": [
        "tool",
        "latency"
      ],
      "latencyThresholdMs": 500
    },
    "proactiveEngagement": true
  }
}


In [4]:
# Create (or update) the agent with Voice Live settings in metadata.
# create_version() is idempotent — re-running produces a new version under the same name.

voice_live_metadata = chunk_config(json.dumps(VOICE_LIVE_CONFIG))

agent = project_client.agents.create_version(
    agent_name=AGENT_NAME,
    definition=PromptAgentDefinition(
        model=MODEL_DEPLOYMENT_NAME,
        instructions=DRIVETHRU_INSTRUCTIONS,
        tools=ALL_TOOLS,
    ),
    metadata=voice_live_metadata,
)
print(f"✅ Agent created")
print(f"   name    : {agent.name}")
print(f"   version : {agent.version}")
print(f"   id      : {agent.id}")

# Verify Voice Live config was stored
retrieved_agent = project_client.agents.get(agent_name=AGENT_NAME)
stored_metadata = (retrieved_agent.versions or {}).get("latest", {}).get("metadata", {})
stored_config = reassemble_config(stored_metadata)
if stored_config:
    print(f"\n✅ Voice Live configuration stored ({len(stored_config)} chars)")
    print(json.dumps(json.loads(stored_config), indent=2))
else:
    print("\n⚠️  Voice Live configuration not found in agent metadata.")

✅ Agent created
   name    : drivethru-ordering-agent
   version : 1
   id      : drivethru-ordering-agent:1

✅ Voice Live configuration stored (627 chars)
{
  "microsoft.voice-live.enabled": "true",
  "session": {
    "voice": {
      "name": "en-US-Ava:DragonHDLatestNeural",
      "type": "azure-standard",
      "temperature": 0.8
    },
    "inputAudioTranscription": {
      "model": "azure-speech",
      "language": "nl-NL,en-US"
    },
    "turnDetection": {
      "type": "azure_semantic_vad",
      "endOfUtteranceDetection": {
        "model": "semantic_detection_v1_multilingual"
      }
    },
    "inputAudioNoiseReduction": {
      "type": "azure_deep_noise_suppression"
    },
    "inputAudioEchoCancellation": {
      "type": "server_echo_cancellation"
    },
    "fillerResponse": {
      "type": "llm_filler",
      "triggers": [
        "tool",
        "latency"
      ],
      "latencyThresholdMs": 500
    },
    "proactiveEngagement": true
  }
}


In [5]:
# ── Next step: update .env and run validation ──────────────────────────────
print("Add/verify the following in your .env file:\n")
print(f'AGENT_NAME="{agent.name}"')
print(f'AGENT_VERSION="{agent.version}"')
print()
print("Then run the voice validation script:")
print("  python -m src.agents.voice_agent_validation")

Add/verify the following in your .env file:

AGENT_NAME="drivethru-ordering-agent"
AGENT_VERSION="1"

Then run the voice validation script:
  python -m src.agents.voice_agent_validation
